# Distances Between Observations

Read this notebook from top to bottom and fill in the code as you go. Work together and discuss with other students in the class. Try to resolve any errors on your own first, but don't get stuck; ask for help!

In addition to writing and running code, be sure to examine any output and interpret the results before moving on.

For many of these questions, there are several approaches, and there is no single right answer. You should try a few different things and compare with your classmates.

We will use `scikit-learn` extensively later, but for this activity you might want to stick with `pandas`.


In [1]:
import pandas as pd
import numpy as np

## Ames - Recommending Similar Homes

1\. Suppose that you really like house 0 in the Ames housing data set, but it is too expensive. Find cheaper homes that are similar to it --- in terms of living area, number of bedrooms, number of bathrooms --- by calculating distances from house 0. You might want to try different distance metrics and different scaling methods; how sensitive are your results to these choices?

Be sure to actually look at the profiles of the homes that your algorithm picked out as most similar (based on these 3 variables). Do they make sense?

**_Think:_ If the goal is to find a "good deal" on a similar house, should sale price be included as a variable in your distance metric?**

If the goal is to find a good deal on a similar house, it does not make sense to include sale price as a variable in my distance metric. This is because adding sale price as a variable would make it so that the function which returns the most similar houses also returns houses that are similar in price, which is not the objective. I would like houses which are similar in all respects except for price if possible.

In [2]:
# YOUR CODE HERE. ADD CELLS AS NEEDED
df_housing = pd.read_csv("https://raw.githubusercontent.com/kevindavisross/data301/main/data/AmesHousing.txt", sep="\t")
df_housing.columns

Index(['Order', 'PID', 'MS SubClass', 'MS Zoning', 'Lot Frontage', 'Lot Area',
       'Street', 'Alley', 'Lot Shape', 'Land Contour', 'Utilities',
       'Lot Config', 'Land Slope', 'Neighborhood', 'Condition 1',
       'Condition 2', 'Bldg Type', 'House Style', 'Overall Qual',
       'Overall Cond', 'Year Built', 'Year Remod/Add', 'Roof Style',
       'Roof Matl', 'Exterior 1st', 'Exterior 2nd', 'Mas Vnr Type',
       'Mas Vnr Area', 'Exter Qual', 'Exter Cond', 'Foundation', 'Bsmt Qual',
       'Bsmt Cond', 'Bsmt Exposure', 'BsmtFin Type 1', 'BsmtFin SF 1',
       'BsmtFin Type 2', 'BsmtFin SF 2', 'Bsmt Unf SF', 'Total Bsmt SF',
       'Heating', 'Heating QC', 'Central Air', 'Electrical', '1st Flr SF',
       '2nd Flr SF', 'Low Qual Fin SF', 'Gr Liv Area', 'Bsmt Full Bath',
       'Bsmt Half Bath', 'Full Bath', 'Half Bath', 'Bedroom AbvGr',
       'Kitchen AbvGr', 'Kitchen Qual', 'TotRms AbvGrd', 'Functional',
       'Fireplaces', 'Fireplace Qu', 'Garage Type', 'Garage Yr Blt',
      

In [3]:
df_housing[['Bsmt Full Bath', 'Bsmt Half Bath', 'Full Bath', 'Half Bath']].describe()

,Bsmt Full Bath,Bsmt Half Bath,Full Bath,Half Bath
count,2928.000000,2928.000000,2930.000000,2930.000000
mean,0.431352,0.061134,1.566553,0.379522
std,0.524820,0.245254,0.552941,0.502629
min,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,1.000000,0.000000
50%,0.000000,0.000000,2.000000,0.000000
75%,1.000000,0.000000,2.000000,1.000000
max,3.000000,2.000000,4.000000,2.000000


In [4]:
df_housing.loc[0]

Order                     1
PID               526301100
MS SubClass              20
MS Zoning                RL
Lot Frontage          141.0
                    ...    
Mo Sold                   5
Yr Sold                2010
Sale Type               WD 
Sale Condition       Normal
SalePrice            215000
Name: 0, Length: 82, dtype: object

In [5]:
df_housing["Total Bathrooms"] = df_housing["Full Bath"] + df_housing["Bsmt Full Bath"] + 0.5 * (df_housing["Half Bath"] + df_housing["Bsmt Half Bath"])
df_housing["Total Bathrooms"].describe().round()

count    2928.0
mean        2.0
std         1.0
min         1.0
25%         2.0
50%         2.0
75%         2.0
max         7.0
Name: Total Bathrooms, dtype: float64

In [6]:
df_housing['Bedroom AbvGr']

0       3
1       2
2       3
3       3
4       3
       ..
2925    3
2926    2
2927    3
2928    2
2929    3
Name: Bedroom AbvGr, Length: 2930, dtype: int64

In [7]:
df_housing['Gr Liv Area']

0       1656
1        896
2       1329
3       2110
4       1629
        ... 
2925    1003
2926     902
2927     970
2928    1389
2929    2000
Name: Gr Liv Area, Length: 2930, dtype: int64

In [8]:
X = df_housing[['Bedroom AbvGr', 'Gr Liv Area', 'Total Bathrooms']]

In [9]:
X_z = (X - X.mean()) / X.std()

In [10]:
X_z.round(2)

,Bedroom AbvGr,Gr Liv Area,Total Bathrooms
0,0.18,0.31,-0.27
1,-1.03,-1.19,-1.51
2,0.18,-0.34,-0.89
3,0.18,1.21,1.59
4,0.18,0.26,0.35
...,...,...,...
2925,0.18,-0.98,-0.27
2926,-1.03,-1.18,-0.27
2927,0.18,-1.05,-0.89
2928,-1.03,-0.22,-0.27


In [11]:
house0 = df_housing.loc[0]
house0[['Bedroom AbvGr', 'Gr Liv Area', 'Total Bathrooms']]

Bedroom AbvGr         3
Gr Liv Area        1656
Total Bathrooms     2.0
Name: 0, dtype: object

In [12]:
euclid_z = np.sqrt(((X_z - X_z.loc[0]) ** 2).sum(axis=1))
euclid_raw = np.sqrt(((X - X.loc[0]) ** 2).sum(axis=1))
manhattan_z = (X_z - X_z.loc[0]).abs().sum(axis=1)

In [13]:
df_three_var = pd.DataFrame({
    "euclidean_raw": euclid_raw,
    "euclidean_z": euclid_z,
    "manhattan_z": manhattan_z,
    "SalePrice": df_housing["SalePrice"]
})

house0_price = df_housing.loc[0, "SalePrice"]

In [14]:
cheaper_homes = df_three_var[(df_three_var["SalePrice"] < house0_price) & (df_three_var.index != 0)]

most_similar_std = cheaper_homes.sort_values("euclidean_z").head(10)
most_similar_std

,euclidean_raw,euclidean_z,manhattan_z,SalePrice
731,2.0,0.003956,0.003956,137000
1226,5.0,0.009891,0.009891,165500
1994,6.0,0.011869,0.011869,144100
2673,8.0,0.015826,0.015826,159000
1940,9.0,0.017804,0.017804,153000
291,10.0,0.019782,0.019782,100000
2637,12.0,0.023738,0.023738,135000
618,12.0,0.023738,0.023738,167000
375,15.0,0.029673,0.029673,200000
2640,15.0,0.029673,0.029673,178000


2\. Continuing part 1. Suppose that you really like house 0 in the data set, but it is too expensive. Find cheaper homes that are similar to it --- in terms of living area, number of bedrooms, number of bathrooms, **and House Style** --- by calculating distances from house 0. You might want to try different distance metrics and different scaling methods; how sensitive are your results to these choices?

Be sure to actually look at the profiles of the homes that your algorithm picked out as most similar. Do they make sense?

Adding house style seems to yield meanignful differences. House 0 is one story, and when I combined house style with the other variables, the top matches also are one story. The closest matches do appear to be reasonable.

In [15]:
# YOUR CODE HERE. ADD CELLS AS NEEDED
house_style_dummies = pd.get_dummies(df_housing["House Style"], prefix="Style", dtype=float)

In [16]:
X_q2 = pd.concat([X_z, house_style_dummies], axis=1)

In [17]:
dist_q2 = np.sqrt(((X_q2 - X_q2.loc[0]) ** 2).sum(axis=1))

In [18]:
df_q2 = pd.DataFrame({"distance": dist_q2, "SalePrice": df_housing["SalePrice"]})
cheaper_q2 = df_q2[(df_q2["SalePrice"] < house0_price) & (df_q2.index != 0)]

most_similar_q2 = cheaper_q2.sort_values("distance").head(10)
most_similar_q2

,distance,SalePrice
1940,0.017804,153000
618,0.023738,167000
375,0.029673,200000
209,0.037586,173000
1185,0.051433,174000
876,0.051433,213000
2075,0.051433,130000
314,0.061324,160000
2282,0.067259,168000
2228,0.079128,198000


In [19]:
df_housing.loc[most_similar_q2.index, ['Bedroom AbvGr', 'Gr Liv Area', 'Total Bathrooms', 'House Style', 'SalePrice', 'Neighborhood']]

,Bedroom AbvGr,Gr Liv Area,Total Bathrooms,House Style,SalePrice,Neighborhood
1940,3,1647,2.0,1Story,153000,NAmes
618,3,1644,2.0,1Story,167000,NAmes
375,3,1671,2.0,1Story,200000,NWAmes
209,3,1675,2.0,1Story,173000,ClearCr
1185,3,1682,2.0,1Story,174000,NWAmes
876,3,1630,2.0,1Story,213000,CollgCr
2075,3,1630,2.0,1Story,130000,Edwards
314,3,1687,2.0,1Story,160000,Timber
2282,3,1622,2.0,1Story,168000,Mitchel
2228,3,1696,2.0,1Story,198000,Crawfor


3\. Continuing parts 1 and 2. Suppose that you really like house 0 in the data set, but it is too expensive. Find cheaper homes that are similar to it, by calculating distances. You can **choose the variables to include, but include both quantitative and categorical variables**. Be sure to actually look at the profiles of the homes that your algorithm picked out as most similar. Do they make sense?

You might want to try different distance metrics and different scaling methods; how sensitive are your results to these choices?

_Hint:_ There are many variables in the data set. Do not attempt to compute distance based on all the variables! You will want to pare down the number of variables, but be sure to include a mixture of categorical and quantitative variables. Refer to the [data documentation](https://ww2.amstat.org/publications/jse/v19n3/decock/DataDocumentation.txt) for information about the variables.


I chose the variables 'Bedroom AbvGr', 'Gr Liv Area', 'Total Bathrooms', 'Overall Qual', 'Neighborhood', and 'SalePrice'. The homes that my algorithm picked out as similar make sense: Bedroom AbvGr, Total Bathrooms, Overall Qual, and Neighborhood are all exactly the same. The other variables (Gr Liv Area and SalePrice) are very similar. I tried a few different scaling methods but it was this method (standardization) where I saw the most success with similarity.

In [20]:
X_q3_quant = df_housing[['Bedroom AbvGr', 'Gr Liv Area', 'Total Bathrooms', 'Overall Qual']]
X_q3_z = (X_q3_quant - X_q3_quant.mean()) / X_q3_quant.std()
X_q3_z

,Bedroom AbvGr,Gr Liv Area,Total Bathrooms,Overall Qual
0,0.176064,0.309212,-0.269988,-0.067242
1,-1.032058,-1.194223,-1.509056,-0.775946
2,0.176064,-0.337661,-0.889522,-0.067242
3,0.176064,1.207317,1.588613,0.641462
4,0.176064,0.255801,0.349546,-0.775946
...,...,...,...,...
2925,0.176064,-0.982555,-0.269988,-0.067242
2926,-1.032058,-1.182354,-0.269988,-0.775946
2927,0.176064,-1.047836,-0.889522,-0.775946
2928,-1.032058,-0.218968,-0.269988,-0.775946


In [21]:
neighborhood_dummies = pd.get_dummies(df_housing['Neighborhood'], prefix='Nbhd', dtype=float)

X_q3 = pd.concat([X_q3_z, neighborhood_dummies], axis=1)
X_q3

,Bedroom AbvGr,Gr Liv Area,Total Bathrooms,Overall Qual,Nbhd_Blmngtn,Nbhd_Blueste,Nbhd_BrDale,Nbhd_BrkSide,Nbhd_ClearCr,Nbhd_CollgCr,...,Nbhd_NoRidge,Nbhd_NridgHt,Nbhd_OldTown,Nbhd_SWISU,Nbhd_Sawyer,Nbhd_SawyerW,Nbhd_Somerst,Nbhd_StoneBr,Nbhd_Timber,Nbhd_Veenker
0,0.176064,0.309212,-0.269988,-0.067242,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,-1.032058,-1.194223,-1.509056,-0.775946,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,0.176064,-0.337661,-0.889522,-0.067242,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,0.176064,1.207317,1.588613,0.641462,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,0.176064,0.255801,0.349546,-0.775946,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2925,0.176064,-0.982555,-0.269988,-0.067242,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2926,-1.032058,-1.182354,-0.269988,-0.775946,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2927,0.176064,-1.047836,-0.889522,-0.775946,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2928,-1.032058,-0.218968,-0.269988,-0.775946,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [22]:
dist_q3 = np.sqrt(((X_q3 - X_q3.loc[0]) ** 2).sum(axis=1))

df_q3 = pd.DataFrame({"distance": dist_q3, "SalePrice": df_housing["SalePrice"]})
cheaper_q3 = df_q3[(df_q3["SalePrice"] < house0_price) & (df_q3.index != 0)]

most_similar_q3 = cheaper_q3.sort_values("distance").head(10)
most_similar_q3

,distance,SalePrice
1226,0.009891,165500
618,0.023738,167000
1611,0.136496,167300
2562,0.154300,133000
1240,0.170126,166800
128,0.197820,172500
122,0.235406,136300
1248,0.265079,180000
132,0.269036,153000
1958,0.375859,143000


In [23]:
df_housing.loc[most_similar_q3.index, ['Bedroom AbvGr', 'Gr Liv Area', 'Total Bathrooms', 'Overall Qual', 'Neighborhood', 'SalePrice']]

,Bedroom AbvGr,Gr Liv Area,Total Bathrooms,Overall Qual,Neighborhood,SalePrice
1226,3,1661,2.0,6,NAmes,165500
618,3,1644,2.0,6,NAmes,167000
1611,3,1587,2.0,6,NAmes,167300
2562,3,1578,2.0,6,NAmes,133000
1240,3,1570,2.0,6,NAmes,166800
128,3,1556,2.0,6,NAmes,172500
122,3,1775,2.0,6,NAmes,136300
1248,3,1790,2.0,6,NAmes,180000
132,3,1520,2.0,6,NAmes,153000
1958,3,1846,2.0,6,NAmes,143000


## Colleges similar to Cal Poly

We'll use data from the [College Scorecard data](https://collegescorecard.ed.gov/) to find colleges and universities that are similar to Cal Poly.

In [24]:
df_college = pd.read_csv("https://datasci112.stanford.edu/data/college_attributes.csv")

df_college.set_index("Institution", inplace = True)

df_college

,City,State,AdmissionRate,Undergraduates,CarnegieClassification,Ownership,PCIP01,PCIP03,PCIP04,PCIP05,...,PCIP44,PCIP45,PCIP46,PCIP47,PCIP48,PCIP49,PCIP50,PCIP51,PCIP52,PCIP54
Institution,,,,,,,,,,,,,,,,,,,,,
Alabama A & M University,Normal,AL,0.7160,5098.0,Master's Colleges & Universities: Larger Programs,Public,0.0445,0.0071,0.0053,0.0000,...,0.0409,0.0249,0.0,0.0,0.0,0.0,0.0231,0.0000,0.1637,0.0000
University of Alabama at Birmingham,Birmingham,AL,0.8854,13284.0,Doctoral Universities: Very High Research Acti...,Public,0.0000,0.0000,0.0000,0.0020,...,0.0195,0.0239,0.0,0.0,0.0,0.0,0.0249,0.2088,0.2159,0.0141
University of Alabama in Huntsville,Huntsville,AL,0.7367,7358.0,Doctoral Universities: Very High Research Acti...,Public,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0127,0.0,0.0,0.0,0.0,0.0407,0.1341,0.1930,0.0073
Alabama State University,Montgomery,AL,0.9799,3495.0,Doctoral/Professional Universities,Public,0.0000,0.0000,0.0000,0.0000,...,0.0648,0.0196,0.0,0.0,0.0,0.0,0.0511,0.0904,0.1513,0.0059
The University of Alabama,Tuscaloosa,AL,0.7890,30725.0,Doctoral Universities: Very High Research Acti...,Public,0.0000,0.0061,0.0000,0.0019,...,0.0072,0.0661,0.0,0.0,0.0,0.0,0.0234,0.1077,0.2916,0.0096
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
Florida Academy of Nursing,Miramar,FL,0.3088,239.0,Not applicable,Private for-profit,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000,1.0000,0.0000,0.0000
Herzing University-Tampa,Tampa,FL,0.9630,68.0,Not applicable,Private nonprofit,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000,0.0000,0.0000,0.0000
Abilene Christian University-Undergraduate Online,Addison,TX,1.0000,415.0,Not applicable,Private nonprofit,0.0000,0.0000,0.0000,0.0000,...,0.0000,0.0000,0.0,0.0,0.0,0.0,0.0000,0.0000,0.0000,0.0000


We'll want to single out Cal Poly, which we can do like this.

In [25]:
school_name = "California Polytechnic State University-San Luis Obispo"

cp = df_college.loc[school_name]

cp

City                                                        San Luis Obispo
State                                                                    CA
AdmissionRate                                                          0.33
Undergraduates                                                      21090.0
CarnegieClassification    Master's Colleges & Universities: Larger Programs
Ownership                                                            Public
PCIP01                                                               0.1084
PCIP03                                                               0.0255
PCIP04                                                               0.0441
PCIP05                                                               0.0019
PCIP09                                                               0.0353
PCIP10                                                               0.0175
PCIP11                                                               0.0326
PCIP12      

1\. Based on only the admission rate and the number of undergraduates, what schools are most similar to Cal Poly? Specify how you're making this decision.

I used standardized Euclidean distance, so that Undergraduates did not totally outweigh Admission Rate due to their differences in scales. I put both variables in standard-deviation units to find the schools whose (admission rate, enrollment) pair is closest to Cal Poly's.

In [26]:
# YOUR CODE HERE. ADD CELLS AS NEEDED
X_college1 = df_college[['AdmissionRate', 'Undergraduates']]

In [27]:
X_college1_z = (X_college1 - X_college1.mean()) / X_college1.std()

dist_college1 = np.sqrt(((X_college1_z - X_college1_z.loc[school_name]) ** 2).sum(axis=1))
dist_college1 = dist_college1.drop(index=school_name)
dist_college1

Institution
Alabama A & M University                             2.700013
University of Alabama at Birmingham                  2.707139
University of Alabama in Huntsville                  2.552062
Alabama State University                             3.712440
The University of Alabama                            2.419517
                                                       ...   
Florida Academy of Nursing                           2.685834
Herzing University-Tampa                             3.940952
Abilene Christian University-Undergraduate Online    4.034715
Great Northern University                            4.068007
Arizona College of Nursing-Salt Lake City            4.054221
Length: 1956, dtype: float64

In [28]:
most_similar_colleges1 = dist_college1.sort_values().head()
most_similar_colleges1

Institution
University of California-Santa Barbara         0.309162
DeVry University-Illinois                      0.593121
University of North Carolina at Chapel Hill    0.596846
Clemson University                             0.736788
University of Virginia-Main Campus             0.761296
dtype: float64

In [29]:
df_college.loc[most_similar_colleges1.index, ['City', 'State', 'AdmissionRate', 'Undergraduates']]

,City,State,AdmissionRate,Undergraduates
Institution,,,,
University of California-Santa Barbara,Santa Barbara,CA,0.2918,23081.0
DeVry University-Illinois,Naperville,IL,0.4552,19729.0
University of North Carolina at Chapel Hill,Chapel Hill,NC,0.2040,19722.0
Clemson University,Clemson,SC,0.4922,21577.0
University of Virginia-Main Campus,Charlottesville,VA,0.2074,17041.0


2\. Now consider the admission rate, the number of undergraduates, and also the [Carnegie classification](https://en.wikipedia.org/wiki/Carnegie_Classification_of_Institutions_of_Higher_Education) of the type of school, and the ownership (public, private, etc.) Based on these variables, what schools are most similar to Cal Poly? Specify how you're making this decision.


I standardized admission rate and undergraduate enrollment, included Carnegie classification and ownership, then measured Euclidean distance from each school to Cal Poly across all four variables. This approach yielded CUNY Hunter College, CUNY Bernard M Baruch College, and CUNY John Jay College of Criminal Justice as the most similar.

In [30]:
categorical_cols = ['CarnegieClassification', 'Ownership']
dummies_college2 = pd.get_dummies(df_college[categorical_cols], dtype=float)
dummies_college2.head()

,CarnegieClassification_Associate's Colleges: High Career & Technical-High Nontraditional,CarnegieClassification_Associate's Colleges: High Career & Technical-High Traditional,CarnegieClassification_Associate's Colleges: High Career & Technical-Mixed Traditional/Nontraditional,CarnegieClassification_Associate's Colleges: High Transfer-High Nontraditional,CarnegieClassification_Associate's Colleges: High Transfer-High Traditional,CarnegieClassification_Associate's Colleges: High Transfer-Mixed Traditional/Nontraditional,CarnegieClassification_Baccalaureate Colleges: Arts & Sciences Focus,CarnegieClassification_Baccalaureate Colleges: Diverse Fields,CarnegieClassification_Baccalaureate/Associate's Colleges: Associate's Dominant,CarnegieClassification_Baccalaureate/Associate's Colleges: Mixed Baccalaureate/Associate's,...,CarnegieClassification_Special Focus Four-Year: Other Special Focus Institutions,CarnegieClassification_Special Focus Four-Year: Research Institution,CarnegieClassification_Special Focus Two-Year: Arts & Design,CarnegieClassification_Special Focus Two-Year: Health Professions,CarnegieClassification_Special Focus Two-Year: Other Fields,CarnegieClassification_Special Focus Two-Year: Technical Professions,CarnegieClassification_Tribal Colleges,Ownership_Private for-profit,Ownership_Private nonprofit,Ownership_Public
Institution,,,,,,,,,,,,,,,,,,,,,
Alabama A & M University,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
University of Alabama at Birmingham,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
University of Alabama in Huntsville,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
Alabama State University,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
The University of Alabama,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0


In [31]:
X_college2 = pd.concat([X_college1_z, dummies_college2], axis=1)

dist_college2 = np.sqrt(((X_college2 - X_college2.loc[school_name]) ** 2).sum(axis=1))
dist_college2 = dist_college2.drop(index=school_name)
dist_college2

Institution
Alabama A & M University                             2.700013
University of Alabama at Birmingham                  3.054276
University of Alabama in Huntsville                  2.917708
Alabama State University                             3.972683
The University of Alabama                            2.802510
                                                       ...   
Florida Academy of Nursing                           3.348687
Herzing University-Tampa                             4.419401
Abilene Christian University-Undergraduate Online    4.503213
Great Northern University                            4.533065
Arizona College of Nursing-Salt Lake City            4.520698
Length: 1956, dtype: float64

In [32]:
most_similar_colleges2 = dist_college2.sort_values().head()

df_college.loc[most_similar_colleges2.index, ['City', 'State', 'AdmissionRate', 'Undergraduates', 'CarnegieClassification', 'Ownership']]

,City,State,AdmissionRate,Undergraduates,CarnegieClassification,Ownership
Institution,,,,,,
CUNY Hunter College,New York,NY,0.4590,17293.0,Master's Colleges & Universities: Larger Programs,Public
CUNY Bernard M Baruch College,New York,NY,0.5056,15483.0,Master's Colleges & Universities: Larger Programs,Public
CUNY John Jay College of Criminal Justice,New York,NY,0.4458,12834.0,Master's Colleges & Universities: Larger Programs,Public
CUNY Brooklyn College,Brooklyn,NY,0.5136,12567.0,Master's Colleges & Universities: Larger Programs,Public
University of California-Santa Barbara,Santa Barbara,CA,0.2918,23081.0,Doctoral Universities: Very High Research Acti...,Public


3\. The columns whose names begin with "PCIP" contain the proportions of students at each school studying various fields (e.g., Engineering, Psychology). Each field is represented by a two-digit code called the [CIP code](https://nces.ed.gov/ipeds/cipcode/browse.aspx?y=55).

If we only consider the proportions of students studying various fields, what schools are most similar to Cal Poly? Specify how you're making this decision.

I made this decision by standardizing the PCIP columns before computing Euclidean distance from Cal Poly. PCIP proportions vary college-by-college, but very popular vs. very niche majors can skew this. Standardization

In [33]:
X_pcip = df_college.filter(like="PCIP").dropna()

In [34]:
X_pcip_z = (X_pcip - X_pcip.mean()) / X_pcip.std()

In [35]:
dist_pcip_z = np.sqrt(((X_pcip_z - X_pcip_z.loc[school_name]) ** 2).sum(axis=1))

In [36]:
most_similar_pcip_z = dist_pcip_z.sort_values().head(10)

In [37]:
most_similar_pcip_z

Institution
California Polytechnic State University-San Luis Obispo             0.000000
Iowa State University                                               1.986207
California State Polytechnic University-Pomona                      2.241726
Texas A & M University-College Station                              2.258942
Mississippi State University                                        2.335288
Clemson University                                                  2.376870
North Carolina State University at Raleigh                          2.508999
West Virginia University                                            2.584194
Louisiana Tech University                                           2.778706
Louisiana State University and Agricultural & Mechanical College    2.802481
dtype: float64

In [38]:
df_college.loc[most_similar_pcip_z.index, ['City', 'State']].join(X_pcip.filter(like="PCIP").iloc[:, :5])

,City,State,PCIP01,PCIP03,PCIP04,PCIP05,PCIP09
Institution,,,,,,,
California Polytechnic State University-San Luis Obispo,San Luis Obispo,CA,0.1084,0.0255,0.0441,0.0019,0.0353
Iowa State University,Ames,IA,0.1013,0.0131,0.0163,0.0009,0.0392
California State Polytechnic University-Pomona,Pomona,CA,0.0384,0.0000,0.0324,0.0043,0.0306
Texas A & M University-College Station,College Station,TX,0.0847,0.0241,0.0111,0.0001,0.0397
Mississippi State University,Mississippi State,MS,0.0619,0.0263,0.0208,0.0000,0.0360
Clemson University,Clemson,SC,0.0628,0.0115,0.0185,0.0029,0.0049
North Carolina State University at Raleigh,Raleigh,NC,0.0906,0.0356,0.0075,0.0000,0.0353
West Virginia University,Morgantown,WV,0.0372,0.0295,0.0000,0.0000,0.0500
Louisiana Tech University,Ruston,LA,0.0305,0.0212,0.0231,0.0000,0.0243
